# Export held-out val/test sets — CAMUS + BRISC, one download

Packages the **validation** and **test** image/mask pairs for both datasets into one zip.

**Why one export covers both Phase A and Phase B:** `splits.py::patient_level_split` carves
out `val_split`/`test_split` from the shuffled patient list *before* `label_frac` is applied —
`label_frac` only shrinks the remaining **train** pool. Phase A (`label_frac=1.0`) and Phase B
(`label_frac=0.05`/`0.10`) use the exact same `seed`, `val_split`, `test_split` (see
`configs/CAMUS/camus.yaml` / `configs/BRISC/brisc.yaml`) and therefore produce the **identical**
val/test patient sets in both phases — this notebook reconstructs that split once per dataset
with `label_frac=1.0` and it is valid for every phase's checkpoints. Same reasoning applies
across the DRL notebooks: they all call `apply_refinement_config` with `baseline_cfg_name`
pointing at this same `camus.yaml`/`brisc.yaml`, so the DRL agents warm-start from — and are
evaluated on — this identical split too. One split, shared by the U-Net baselines and every DRL
agent, both phases.

**`val` vs `test` — read before using either for a paper claim:**
- **`test`** — never touched by anything, at any point (no gradient updates, no checkpoint
  selection, no early-stopping decision). The strict "never seen" set.
- **`val`** — never trained on (no gradient updates), but it DOES drive checkpoint/model
  selection (best-val-Dice picking, early stopping) for both the U-Net baselines and every DRL
  agent. Not leaked in the traditional sense, but not fully untouched either.

Exported separately (`val/` and `test/` subfolders) so you can choose per use-case — e.g. use
`test` for any qualitative figure or held-out number that needs the strongest "never seen"
claim.

**Requires:** the `iteris-pkg`, `CAMUS`, and `brisc2025` datasets attached (same as any other
notebook in this project). Missing one dataset doesn't block the other — each is processed
independently and skipped with a warning if its data root isn't found.

## 0 · Install + locate package

In [ ]:
import subprocess, sys
from pathlib import Path

init_files = list(Path('/kaggle/input').rglob('iteris/__init__.py'))
if not init_files:
    raise RuntimeError('iteris-pkg not attached. Add it in the right-hand panel → Datasets.')
PKG_ROOT = init_files[0].parent.parent
subprocess.run(['pip', 'install', '-r', str(PKG_ROOT / 'requirements.txt'),
                '--quiet', '--upgrade'], check=True)
sys.path.insert(0, str(PKG_ROOT))
print(f'✓ Package at: {PKG_ROOT}')

## 1 · Locate both dataset roots (auto-detect, same rglob pattern every other notebook uses)

In [ ]:
from iteris.config import load_config

DATASETS = {}

camus_cfg = load_config(str(PKG_ROOT / 'configs' / 'CAMUS' / 'camus.yaml'))
camus_dirs = [p for p in Path('/kaggle/input').rglob('CAMUS') if p.is_dir()]
if camus_dirs:
    camus_cfg['data_root'] = str(camus_dirs[0])
    DATASETS['CAMUS'] = camus_cfg
    print(f'✓ CAMUS data root : {camus_cfg["data_root"]}')
else:
    print('[WARN] CAMUS dataset not attached — skipping CAMUS export.')

brisc_cfg = load_config(str(PKG_ROOT / 'configs' / 'BRISC' / 'brisc.yaml'))
brisc_dirs = [p for p in Path('/kaggle/input').rglob('brisc2025') if p.is_dir()]
if not brisc_dirs:
    brisc_dirs = [p for p in Path('/kaggle/input').rglob('BRISC') if p.is_dir()]
if brisc_dirs:
    brisc_cfg['data_root'] = str(brisc_dirs[0])
    DATASETS['BRISC'] = brisc_cfg
    print(f'✓ BRISC data root : {brisc_cfg["data_root"]}')
else:
    print('[WARN] BRISC dataset not attached — skipping BRISC export.')

if not DATASETS:
    raise RuntimeError('Neither CAMUS nor BRISC data root found — attach at least one dataset.')

for name, cfg in DATASETS.items():
    print(f'{name}: seed={cfg["seed"]}  val_split={cfg["val_split"]}  test_split={cfg["test_split"]}')

## 2 · Rebuild the exact patient-level split every training run uses

`label_frac=1.0` explicitly — irrelevant to val/test (see title cell), fixed here so this cell
never accidentally shrinks anything. `train` is discarded; this notebook only exports held-out
data.

In [ ]:
from iteris.ingestion import build_dataset_dicts
from iteris.splits import patient_level_split

HOLDOUT = {}   # name -> {'val': [...], 'test': [...]}

for name, cfg in DATASETS.items():
    print(f'--- {name} ---')
    records = build_dataset_dicts(cfg)
    train, val, test = patient_level_split(
        records,
        val_split=cfg['val_split'],
        test_split=cfg['test_split'],
        label_frac=1.0,
        seed=cfg['seed'],
    )
    HOLDOUT[name] = {'val': val, 'test': test}
    print(f'{name}: val={len(val)} samples across {len({r["patient"] for r in val})} patients | '
          f'test={len(test)} samples across {len({r["patient"] for r in test})} patients '
          f'(train={len(train)}, discarded here — this notebook only exports held-out data)')
    print()

## 3 · Sanity check — no patient leakage between splits

Patient-level splitting guarantees this by construction (`splits.py`), but verify directly
rather than trust it blindly — this is the property that makes "held-out" a meaningful claim.

In [ ]:
for name, cfg in DATASETS.items():
    records = build_dataset_dicts(cfg)
    train, val, test = patient_level_split(
        records, val_split=cfg['val_split'], test_split=cfg['test_split'],
        label_frac=1.0, seed=cfg['seed'])
    train_p = {r['patient'] for r in train}
    val_p   = {r['patient'] for r in val}
    test_p  = {r['patient'] for r in test}
    overlap = (train_p & val_p) | (train_p & test_p) | (val_p & test_p)
    assert not overlap, f'{name}: patient leakage across splits: {overlap}'
    print(f'✓ {name}: {len(train_p)} train / {len(val_p)} val / {len(test_p)} test patients — no overlap')

## 4 · Copy val + test image/mask pairs to disk, with a provenance manifest

Layout: `/kaggle/working/holdout/<DATASET>/<val|test>/{images,masks}/<original filename>`

Each dataset also gets a `<dataset>_<split>_manifest.csv` recording patient ID (and
view/phase for CAMUS, tumor_type for BRISC) alongside the exported filename — keep this if you
cite the held-out set in the paper, it's the reproducibility trail.

In [ ]:
import shutil
import pandas as pd

OUT_ROOT = Path('/kaggle/working/holdout')
if OUT_ROOT.exists():
    shutil.rmtree(OUT_ROOT)

for name, splits in HOLDOUT.items():
    for split_name, recs in splits.items():
        img_dir = OUT_ROOT / name / split_name / 'images'
        msk_dir = OUT_ROOT / name / split_name / 'masks'
        img_dir.mkdir(parents=True, exist_ok=True)
        msk_dir.mkdir(parents=True, exist_ok=True)

        manifest_rows = []
        for r in recs:
            img_src, lbl_src = Path(r['image']), Path(r['label'])
            img_dst = img_dir / img_src.name
            msk_dst = msk_dir / lbl_src.name
            shutil.copy2(img_src, img_dst)
            shutil.copy2(lbl_src, msk_dst)
            row = {'patient': r['patient'], 'image_file': img_dst.name, 'mask_file': msk_dst.name}
            if 'view' in r:        row['view'] = r['view']
            if 'phase' in r:       row['phase'] = r['phase']
            if 'tumor_type' in r:  row['tumor_type'] = r['tumor_type']
            manifest_rows.append(row)

        manifest_path = OUT_ROOT / name / f'{name.lower()}_{split_name}_manifest.csv'
        pd.DataFrame(manifest_rows).to_csv(manifest_path, index=False)
        print(f'{name}/{split_name}: {len(recs)} pairs copied -> {img_dir.parent}  '
              f'(manifest: {manifest_path.name})')

## 5 · Zip for one-click download

In [ ]:
zip_path = shutil.make_archive('/kaggle/working/iteris_holdout_val_test', 'zip', OUT_ROOT)
size_mb = Path(zip_path).stat().st_size / (1024 * 1024)
print(f'✓ Zipped -> {zip_path}  ({size_mb:.1f} MB)')
print('Download from the notebook\'s Output panel (right sidebar) once this cell finishes.')